# LLM API on Colab (temporary, for testing)

Runs an open model with [Ollama](https://ollama.com) on Colab's free T4 GPU and gives you a
**temporary, key-protected, OpenAI-compatible API URL** you can call from your own code.

**How to use**
1. `Runtime → Change runtime type → T4 GPU`.
2. Pick a model and a time limit in cell 1, then `Runtime → Run all`.
3. Cell 2 prints the base URL and API key after a few minutes. Cell 3 keeps the API alive and
   shuts it down at the time limit. **Stop cell 3 to shut down early.**

**Good to know**
- The URL changes every run and dies when the runtime stops. It's for testing, not for anything
  that needs a stable endpoint.
- A request without the key gets `401`. The key is printed in the output: clear outputs
  (`Edit → Clear all outputs`) before sharing or saving this notebook anywhere public.
- Keep the browser tab open. Free Colab can still disconnect you, ask "Are you still there?",
  or refuse a GPU when you've used a lot recently.
- Colab's terms forbid using it to host a web service for others, and using several accounts to
  get around its limits. Use this only for your own short, interactive testing.

In [ ]:
#@title 1. Settings
MODEL = "qwen2.5:7b-instruct-q4_K_M"  #@param ["qwen2.5:7b-instruct-q4_K_M", "qwen3:8b", "llama3.1:8b", "gemma3:4b"] {allow-input: true}
HOURS = 2  #@param {type:"number"}
CONTEXT_LENGTH = 8192  #@param {type:"integer"}
RELEASE_GPU_WHEN_DONE = True  #@param {type:"boolean"}
#@markdown ---
#@markdown Advanced
PROXY_PORT = 8787  #@param {type:"integer"}
CLOUDFLARED_VERSION = "2026.11.0"  #@param {type:"string"}

In [ ]:
#@title 2. Start the API (takes a few minutes)
import json
import os
import platform
import re
import secrets
import shutil
import subprocess
import sys
import time
import urllib.request

WORK = "/content/llm-api"
OLLAMA = "http://127.0.0.1:11434"
PROXY_SRC = '\nimport hmac\nimport os\n\nfrom aiohttp import ClientSession, ClientTimeout, web\n\nKEY = os.environ.pop("LLM_API_KEY")\nUPSTREAM = os.environ["UPSTREAM"]\nPORT = int(os.environ["PROXY_PORT"])\n# Hop-by-hop headers and our own auth header are not forwarded.\nDROP = {\n    "authorization", "host", "connection", "keep-alive", "proxy-authenticate",\n    "proxy-authorization", "te", "trailers", "transfer-encoding", "upgrade", "content-length",\n}\n\n\n@web.middleware\nasync def auth(request, handler):\n    got = request.headers.get("Authorization", "").encode()\n    if not hmac.compare_digest(got, f"Bearer {KEY}".encode()):\n        return web.json_response({"error": "unauthorized"}, status=401)\n    return await handler(request)\n\n\nasync def health(request):\n    try:\n        async with request.app["http"].get(f"{UPSTREAM}/api/tags",\n                                           timeout=ClientTimeout(total=5)) as r:\n            ok = r.status == 200\n    except Exception:\n        ok = False\n    return web.json_response({"ok": ok}, status=200 if ok else 503)\n\n\nasync def forward(request):\n    """Forwards to Ollama and streams the response chunk by chunk, without buffering."""\n    headers = {k: v for k, v in request.headers.items() if k.lower() not in DROP}\n    body = await request.read()\n    try:\n        upstream = await request.app["http"].request(\n            request.method, f"{UPSTREAM}{request.rel_url}", headers=headers, data=body or None)\n    except Exception as e:\n        return web.json_response({"error": f"model server: {type(e).__name__}"}, status=502)\n    async with upstream:\n        resp = web.StreamResponse(status=upstream.status)\n        for k, v in upstream.headers.items():\n            if k.lower() not in DROP:\n                resp.headers[k] = v\n        await resp.prepare(request)\n        async for chunk in upstream.content.iter_any():\n            await resp.write(chunk)\n        await resp.write_eof()\n        return resp\n\n\nasync def make_app():\n    app = web.Application(middlewares=[auth], client_max_size=64 * 1024 * 1024)\n    app["http"] = ClientSession(timeout=ClientTimeout(total=None, sock_connect=10))\n\n    async def close(app):\n        await app["http"].close()\n\n    app.on_cleanup.append(close)\n    app.router.add_get("/health", health)\n    app.router.add_route("*", "/{tail:.*}", forward)\n    return app\n\n\nweb.run_app(make_app(), host="127.0.0.1", port=PORT, access_log=None, print=None)\n'
os.makedirs(WORK, exist_ok=True)


def step(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def http_ok(url, headers=None, timeout=5):
    try:
        req = urllib.request.Request(url, headers=headers or {})
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status == 200
    except Exception:
        return False


def wait_until(check, timeout_s, what):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if check():
            return
        time.sleep(2)
    raise RuntimeError(f"{what} did not come up within {timeout_s}s (logs in {WORK})")


def start(name, args, env=None):
    log = open(os.path.join(WORK, f"{name}.log"), "w")
    PROCS[name] = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT,
                                   stdin=subprocess.DEVNULL, env=env, start_new_session=True)


def stop_all():
    for name, p in list(PROCS.items()):
        if p.poll() is None:
            p.terminate()
            try:
                p.wait(timeout=10)
            except subprocess.TimeoutExpired:
                p.kill()
        del PROCS[name]


# Re-running this cell replaces the previous server.
PROCS = globals().get("PROCS", {})
stop_all()

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No GPU. Use Runtime -> Change runtime type -> T4 GPU, then run again.")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True, check=True).stdout.strip()
step(f"GPU: {gpu}")

if shutil.which("ollama") is None:
    # Ollama's installer needs zstd, which the Colab image doesn't have.
    if shutil.which("zstd") is None:
        step("installing zstd")
        subprocess.run("apt-get update -qq && apt-get install -y -qq zstd", shell=True,
                       check=True, stdout=subprocess.DEVNULL)
    step("installing Ollama")
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "aiohttp"], check=True)

step("starting Ollama")
start("ollama", ["ollama", "serve"], env=dict(
    os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_KEEP_ALIVE="-1",
    OLLAMA_CONTEXT_LENGTH=str(CONTEXT_LENGTH)))
wait_until(lambda: http_ok(f"{OLLAMA}/api/tags"), 120, "Ollama")

step(f"downloading {MODEL}")
with open(os.path.join(WORK, "pull.log"), "w") as log:
    if subprocess.run(["ollama", "pull", MODEL], stdout=log, stderr=subprocess.STDOUT).returncode:
        raise RuntimeError(f"could not download {MODEL}: see {WORK}/pull.log")

step("loading the model into GPU memory")
warm = json.dumps({"model": MODEL, "messages": [{"role": "user", "content": "hi"}],
                   "stream": False}).encode()
urllib.request.urlopen(urllib.request.Request(
    f"{OLLAMA}/api/chat", data=warm, headers={"Content-Type": "application/json"}),
    timeout=600).read()

API_KEY = "sk-" + secrets.token_urlsafe(24)
AUTH = {"Authorization": f"Bearer {API_KEY}"}
with open(os.path.join(WORK, "proxy.py"), "w") as f:
    f.write(PROXY_SRC)
# The key goes through the environment, never the command line.
start("proxy", [sys.executable, os.path.join(WORK, "proxy.py")], env=dict(
    os.environ, LLM_API_KEY=API_KEY, UPSTREAM=OLLAMA, PROXY_PORT=str(PROXY_PORT)))
wait_until(lambda: http_ok(f"http://127.0.0.1:{PROXY_PORT}/health", AUTH), 30, "the auth proxy")

cloudflared = os.path.join(WORK, "cloudflared")
if not os.path.exists(cloudflared):
    step("downloading cloudflared")
    arch = {"x86_64": "amd64", "aarch64": "arm64"}[platform.machine()]
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/download/"
        f"{CLOUDFLARED_VERSION}/cloudflared-linux-{arch}", cloudflared)
    os.chmod(cloudflared, 0o755)

step("opening the tunnel")
start("cloudflared", [cloudflared, "tunnel", "--no-autoupdate",
                      "--url", f"http://127.0.0.1:{PROXY_PORT}"])
URL = None


def tunnel_url():
    global URL
    with open(os.path.join(WORK, "cloudflared.log")) as f:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", f.read())
    URL = m.group(0) if m else None
    return URL is not None


wait_until(tunnel_url, 60, "the tunnel URL")
# A new quick-tunnel URL takes around 30 seconds before it answers (DNS).
step("waiting for the URL to start answering (about 30 seconds)")
wait_until(lambda: http_ok(f"{URL}/health", AUTH), 180, "the public URL")

DEADLINE = time.time() + HOURS * 3600
print(f"""
API ready. Now run cell 3 to keep it alive.

  Base URL : {URL}/v1      (OpenAI-compatible; Ollama's own API is at {URL}/api)
  API key  : {API_KEY}
  Model    : {MODEL}
  Shuts down at {time.strftime('%H:%M', time.localtime(DEADLINE))} (runtime clock, UTC)

Try it:
  curl {URL}/v1/chat/completions \\
    -H "Authorization: Bearer {API_KEY}" -H "Content-Type: application/json" \\
    -d '{{"model": "{MODEL}", "messages": [{{"role": "user", "content": "Hello"}}]}}'
""")

In [ ]:
#@title 3. Keep the API alive (leave this running; stop it to shut down)
import time

try:
    while time.time() < DEADLINE:
        dead = [name for name, p in PROCS.items() if p.poll() is not None]
        if dead:
            print(f"\nStopped unexpectedly: {', '.join(dead)}. Logs are in {WORK}.")
            break
        left = int((DEADLINE - time.time()) // 60)
        print(f"{time.strftime('%H:%M')}  API up, {left} min left", flush=True)
        time.sleep(min(300, max(1, DEADLINE - time.time())))
    else:
        print("Time limit reached.")
except KeyboardInterrupt:
    print("Stopped by you.")
finally:
    stop_all()
    print("API shut down.")
    if RELEASE_GPU_WHEN_DONE:
        print("Releasing the GPU runtime.")
        from google.colab import runtime
        runtime.unassign()

## Calling the API

Any OpenAI-compatible client works. Use the base URL and key printed by cell 2:

```python
from openai import OpenAI

client = OpenAI(base_url="https://<your-url>.trycloudflare.com/v1", api_key="sk-...")
stream = client.chat.completions.create(
    model="qwen2.5:7b-instruct-q4_K_M",
    messages=[{"role": "user", "content": "Write a haiku about GPUs"}],
    stream=True,
)
for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="")
```

Ollama's native API (`/api/chat`, `/api/generate`, `/api/embed`) is available at the same URL
with the same `Authorization: Bearer <key>` header.

**Troubleshooting**: logs are in `/content/llm-api/` (`ollama.log`, `pull.log`, `proxy.log`,
`cloudflared.log`). If cell 2 fails at the GPU step, check the runtime type, or wait if Colab
refused a GPU because of recent heavy use.